# 📊 Notebook 1 — EDA & Preprocessing (Grade Prediction Version)
### Student Performance Prediction — Multi-Class Edition
**Owner:** Person 1

**What changed from v1:**
- Target is now a letter grade (A/B/C/D/E/F) instead of pass/fail
- Everything else (cleaning, encoding, scaling) is identical

**What this notebook produces:**
- `X_train_v2.csv`, `X_test_v2.csv`, `y_train_v2.csv`, `y_test_v2.csv`

## Step 1 — Install & Import Libraries

In [ ]:
!pip install shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## Step 2 — Download the Dataset

In [ ]:
# Download the UCI Student Performance dataset (Math subject)
# The separator in this file is semicolon (;) not comma
!wget -q https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv -O student-mat.csv

df = pd.read_csv('student-mat.csv', sep=';')
print(f'Dataset loaded: {df.shape[0]} students, {df.shape[1]} columns')
df.head()

## Step 3 — Data Quality Check

In [ ]:
# Check for missing values and duplicates
# A clean dataset has 0 of both
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nG3 (final grade) range: {df["G3"].min()} to {df["G3"].max()}')

## Step 4 — Create Letter Grade Target Variable

In [ ]:
# KEY DIFFERENCE FROM V1:
# Instead of binary pass/fail, we map G3 (0-20) to 6 letter grades
# This is a standard Portuguese grading scale adapted to A-F:
#
#   18-20 → A  (Excellent)
#   15-17 → B  (Good)
#   12-14 → C  (Satisfactory)
#   10-11 → D  (Sufficient / just passing)
#    8- 9 → E  (Poor / borderline)
#    0- 7 → F  (Fail)

def assign_grade(g3):
    if g3 >= 18:   return 'A'
    elif g3 >= 15: return 'B'
    elif g3 >= 12: return 'C'
    elif g3 >= 10: return 'D'
    elif g3 >= 8:  return 'E'
    else:          return 'F'

df['grade'] = df['G3'].apply(assign_grade)

# Show distribution of grades
grade_order = ['A', 'B', 'C', 'D', 'E', 'F']
grade_counts = df['grade'].value_counts().reindex(grade_order)

print('Grade distribution:')
for grade, count in grade_counts.items():
    bar = '█' * count
    print(f'  {grade}: {count:>3} students  {bar}')

# Visualize grade distribution
colors = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#e67e22', '#e74c3c']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

grade_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Grade Distribution (Count)', fontsize=12)
axes[0].set_xlabel('Grade')
axes[0].set_ylabel('Number of Students')
axes[0].tick_params(axis='x', rotation=0)

grade_counts.plot(kind='pie', ax=axes[1], colors=colors,
                  autopct='%1.1f%%', startangle=90)
axes[1].set_title('Grade Distribution (%)', fontsize=12)
axes[1].set_ylabel('')

plt.suptitle('Target Variable: Letter Grade Distribution', fontsize=13)
plt.tight_layout()
plt.savefig('grade_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nSaved grade_distribution.png ✅')

## Step 5 — EDA: Grade vs Key Features

In [ ]:
# Box plots show how each numerical feature differs across grade groups
# This helps us see which features clearly separate the grade levels

key_features = ['studytime', 'failures', 'absences', 'goout', 'Walc', 'Medu']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    # Order grades so x-axis goes A → F
    df.boxplot(column=feat, by='grade', ax=axes[i],
               order=grade_order, grid=False)
    axes[i].set_title(f'{feat} by Grade', fontsize=10)
    axes[i].set_xlabel('Grade')
    axes[i].set_ylabel(feat)

plt.suptitle('Key Features vs Letter Grade', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('features_vs_grade.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved features_vs_grade.png ✅')

## Step 6 — EDA: Correlation Heatmap

In [ ]:
# Heatmap of correlations between numerical features
# Values close to +1 or -1 mean strong relationship
# We include G3 here to see what correlates with the raw grade

num_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
            'failures', 'famrel', 'freetime', 'goout',
            'Dalc', 'Walc', 'health', 'absences', 'G3']

plt.figure(figsize=(12, 9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numerical Features', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap_v2.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved correlation_heatmap_v2.png ✅')

## Step 7 — Preprocessing: Drop Leaky Columns & Encode

In [ ]:
# Drop G1, G2, G3 to prevent data leakage
# G3 IS our target. G1 and G2 are intermediate grades that directly
# determine G3 — a real system wouldn't have these at prediction time
df_model = df.drop(columns=['G1', 'G2', 'G3'])

# One-Hot Encode all categorical text columns
# e.g. 'sex' (M/F) → 'sex_M' (1=Male, 0=Female)
# drop_first=True avoids multicollinearity (dummy variable trap)
df_encoded = pd.get_dummies(df_model, drop_first=True)

print(f'Shape before encoding: {df_model.shape}')
print(f'Shape after encoding:  {df_encoded.shape}')

## Step 8 — Train-Test Split & Scaling

In [ ]:
# Separate features (X) and target (y)
X = df_encoded.drop(columns=['grade'])
y = df_encoded['grade']

# 80/20 split — stratify ensures all 6 grade classes appear in both sets
# Without stratify, a rare grade like A might end up only in train or only in test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler: normalize all features to mean=0, std=1
# Fit ONLY on training data — never let test data influence the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {X_train_scaled.shape[0]}')
print(f'Testing samples  : {X_test_scaled.shape[0]}')
print(f'\nGrade distribution in training set:')
print(y_train.value_counts().reindex(grade_order))
print(f'\nGrade distribution in test set:')
print(y_test.value_counts().reindex(grade_order))

## Step 9 — Save Processed Data for Person 2

In [ ]:
# Save as CSVs with _v2 suffix to distinguish from the binary version
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_df  = pd.DataFrame(X_test_scaled,  columns=X.columns)

X_train_df.to_csv('X_train_v2.csv', index=False)
X_test_df.to_csv('X_test_v2.csv',   index=False)
y_train.to_csv('y_train_v2.csv',    index=False)
y_test.to_csv('y_test_v2.csv',      index=False)

print('Files saved — download and share with Person 2:')
print('  ✅ X_train_v2.csv')
print('  ✅ X_test_v2.csv')
print('  ✅ y_train_v2.csv')
print('  ✅ y_test_v2.csv')
print('\nDownload from the Files panel (folder icon) on the left in Colab')